# 🏆 LifeMate - Zalo AI Challenge 2026 (Kaggle 2x Tesla T4 GPUs)

Notebook này được thiết kế riêng để chạy pipeline Agentic Voice Assistant trên **Kaggle Notebooks với 2x GPU Tesla T4 (Tổng cộng ~30GB VRAM)**.

---

### ⚡ Tối ưu song song 2 GPU (Data Parallelism):
- **Cả 2 GPU hoạt động 100% công suất đồng thời**: GPU 0 xử lý nửa đầu (Shard 0), GPU 1 xử lý nửa sau (Shard 1).
- **Tốc độ nhanh gấp đôi**: Toàn bộ 634 mẫu `public_test` chỉ mất **khoảng 15 - 20 phút**!
- **Không lo OOM**: Mỗi GPU có 15GB VRAM, mô hình 4-bit chỉ chiếm ~2.8GB (dư tới 12GB VRAM mỗi GPU).

---

### ⚙️ Thiết lập quan trọng trên Kaggle trước khi chạy:
Ở thanh menu bên phải (**Session options**):
1. **ACCELERATOR**: Chọn **`GPU T4 x2`**
2. **INTERNET**: Bật **`Internet On`** (Bắt buộc bật để clone Git và tải weights mô hình)
3. **INPUT**: Nhấn **`+ Add Data`** để thêm dataset `public_test` mà bạn đã upload lên Kaggle Datasets.

## Bước 1: Kiểm tra 2x GPU Tesla T4

In [ ]:
# Kiểm tra cả 2 GPU Tesla T4 (GPU 0 và GPU 1)
import torch
print(f"🔥 Phát hiện {torch.cuda.device_count()} GPU:")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"   • GPU {i}: {p.name} ({p.total_memory / (1024**3):.1f} GB VRAM)")

!nvidia-smi

## Bước 2: Clone Codebase từ Git Repo

In [ ]:
import os

# Di chuyển vào thư mục làm việc ghi được (/kaggle/working)
%cd /kaggle/working
!rm -rf LifeMate

# Thay <URL_REPO_CỦA_BẠN> bằng URL git của bạn
REPO_URL = "https://github.com/your-username/LifeMate.git"  # <-- Cập nhật URL repo của bạn
BRANCH = "colab-t4"                                         # <-- Branch mới nhất

!git clone -b $BRANCH $REPO_URL LifeMate
%cd /kaggle/working/LifeMate

## Bước 3: Cài đặt thư viện phụ thuộc

In [ ]:
# Cài đặt các package cần thiết: transformers, accelerate, bitsandbytes, qwen-agent, sherpa-onnx
!pip install -q -r requirements.txt

## Bước 4: Tự động tìm đường dẫn Dataset trong `/kaggle/input`

In [ ]:
from pathlib import Path

# Quét thư mục chứa public_test trong /kaggle/input
kaggle_input = Path("/kaggle/input")
found_dirs = list(kaggle_input.rglob("public_test.jsonl"))

if found_dirs:
    data_dir = str(found_dirs[0].parent)
    print(f"✅ Đã tìm thấy thư mục dữ liệu public_test: {data_dir}")
else:
    # Fallback thủ công nếu bạn đặt tên dataset khác
    data_dir = "/kaggle/input/zaic2026-public-test/public_test"  # <-- Sửa đường dẫn nếu cần
    print(f"⚠️ Không tự động tìm thấy, sử dụng đường dẫn mặc định: {data_dir}")

os.environ["DATA_DIR"] = data_dir

## Bước 5: Chạy thử nghiệm 2 mẫu đầu tiên (Sanity Check)
Chạy thử 2 mẫu để xác nhận môi trường và inference trơn tru.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!python main.py \
    --mode public_test \
    --num-samples 2 \
    --data-dir "$DATA_DIR" \
    --output /kaggle/working/test_submission.jsonl

## Bước 6: Chạy toàn bộ 634 mẫu public_test song song trên 2 GPU (KHUYÊN DÙNG)
Chạy đồng thời 2 tiến trình độc lập trên GPU 0 và GPU 1 (Data Parallelism):
- **GPU 0**: Xử lý Shard 0 (317 mẫu) -> lưu vào `/kaggle/working/sub_part0.jsonl`
- **GPU 1**: Xử lý Shard 1 (317 mẫu) -> lưu vào `/kaggle/working/sub_part1.jsonl`
- Cả **2 thanh GPU Utilization sẽ đạt 100% liên tục**, thời gian chạy giảm còn **~15 - 20 phút**!

In [ ]:
%%bash
# Chạy song song 2 worker trên GPU 0 và GPU 1
CUDA_VISIBLE_DEVICES=0 python main.py \
    --mode public_test \
    --num-samples -1 \
    --data-dir "$DATA_DIR" \
    --num-shards 2 \
    --shard-id 0 \
    --output /kaggle/working/sub_part0.jsonl > /kaggle/working/gpu0.log 2>&1 &

CUDA_VISIBLE_DEVICES=1 python main.py \
    --mode public_test \
    --num-samples -1 \
    --data-dir "$DATA_DIR" \
    --num-shards 2 \
    --shard-id 1 \
    --output /kaggle/working/sub_part1.jsonl > /kaggle/working/gpu1.log 2>&1 &

# Chờ cả 2 GPU hoàn thành
wait

# Gộp kết quả 2 phần thành submission.jsonl hoàn chỉnh
cat /kaggle/working/sub_part0.jsonl /kaggle/working/sub_part1.jsonl > /kaggle/working/submission.jsonl
echo "🎉 Hoàn thành xử lý toàn bộ 634 mẫu trên cả 2 GPU!"

## Bước 7: Theo dõi tiến độ chạy của 2 GPU (Chạy cell này bất kỳ lúc nào)

In [ ]:
print("=== Tiến độ GPU 0 ===")
!tail -n 5 /kaggle/working/gpu0.log 2>/dev/null || echo "GPU 0 đang khởi động..."

print("\n=== Tiến độ GPU 1 ===")
!tail -n 5 /kaggle/working/gpu1.log 2>/dev/null || echo "GPU 1 đang khởi động..."

# Kiểm tra số dòng tạm thời
!wc -l /kaggle/working/sub_part0.jsonl /kaggle/working/sub_part1.jsonl 2>/dev/null || true

## Bước 8: Kiểm tra kết quả hoàn thành & Nộp bài
Sau khi Bước 6 chạy xong, file `submission.jsonl` sẽ nằm tại `/kaggle/working/submission.jsonl`. Bạn có thể tải trực tiếp từ tab **Output** bên phải màn hình Kaggle.

In [ ]:
# Kiểm tra số dòng (phải đủ 634 mẫu)
!wc -l /kaggle/working/submission.jsonl

# Xem thử 3 dòng đầu
!head -n 3 /kaggle/working/submission.jsonl

print("✅ File submission.jsonl đã sẵn sàng để nộp bài!")